# 08 — Integración interna: un modelo relacional para analizar
Usa solo los doce Parquet de `data/processed/` y publica `data/processed/analitico/`:

| tabla | grano | clave |
|---|---|---|
| `dataset_analitico_interno` | detalle territorial de causas por proceso (1.997 filas) | ámbito + territorio + materia homologada + tipo de proceso + etapa + contexto penal |
| `metricas_tipo_proceso_long` | familia × fila fuente × columna-métrica | familia + cuadro + página + orden de fila + columna |
| `movimiento_gestion_2023` | ámbito × materia × 2023 | ámbito + materia homologada + gestión |
| `recursos_judiciales_geografia` | ámbito × territorio | ámbito + territorio |
| `personal_geografia` | distrito judicial | departamento |

La tabla principal solo recibe uniones **N:1** validadas (recursos y personal). Las métricas de otro
grano quedan en tablas aparte: copiarlas sobre cada proceso multiplicaría filas y sumas (*fan-out*).
Ninguna ausencia se convierte en cero.

In [ ]:
%run -i modulos/comun.ipynb
%run -i modulos/geografia.ipynb
import hashlib
import matplotlib.pyplot as plt

ANALITICO = PROCESSED / "analitico"
AUDITORIA = PROCESSED / "auditoria"

FILAS_ORIGINALES = {
    "causas_movimiento": 78, "causas_por_gestion": 235, "causas_serie_historica": 51, "juzgados": 1148,
    "personal": 85, "personal_jurisdiccional": 3, "autoridad_sumariante": 27, "causas_por_tipo_proceso": 2419,
    "resueltas_por_tipo_proceso": 27993, "apelaciones_por_tipo_proceso": 37871,
    "ejecucion_por_tipo_proceso": 10015, "otros_tramites_por_tipo_proceso": 6028,
}
FAMILIAS_METRICAS = {
    "resueltas_por_tipo_proceso": "resueltas",
    "apelaciones_por_tipo_proceso": "apelaciones",
    "ejecucion_por_tipo_proceso": "ejecucion",
    "otros_tramites_por_tipo_proceso": "otros_tramites",
}
CLAVE_PRINCIPAL = ["ambito", "territorio", "materia_homologada", "tipo_proceso", "etapa_proceso_fuente", "contexto_accion_penal"]
CLAVE_LONGITUDINAL = ["familia_metrica", "cuadro_origen", "pagina_pdf", "orden_fila", "columna"]
ACCIONES_PENALES_AUDITADAS = {"ACCIÓN PENAL PÙBLICA", "ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE", "ACCIÓN PENAL PRIVADA"}
COMPONENTES_RECURSOS = ["juzgados_publicados", "tribunales_publicados", "salas_publicadas", "conciliadores_publicados", "otros_organos_publicados"]
CODIGOS_TRIBUNAL = {"tribunal_sentencia", "tribunal_sentencia_ampliacion_competencias", "tribunal_sentencia_nominal"}
CODIGOS_OTRO_ORGANO = {"ejecucion_penal"}
SUMAS_BASE_ESPERADAS = {"nuevas_ingresadas": 378684, "atendidas": 706485, "resueltas": 212499,
                        "pendientes_fin": 311478, "pendientes_inicio": 252037}
GRANOS_ANALITICOS = {
    "dataset_analitico_interno": "ámbito × territorio × materia homologada × literal de proceso × etapa fuente × contexto penal",
    "metricas_tipo_proceso_long": "familia de métrica × cuadro × fila fuente × columna-métrica",
    "movimiento_gestion_2023": "ámbito × materia homologada × gestión 2023",
    "recursos_judiciales_geografia": "ámbito × territorio",
    "personal_geografia": "distrito judicial/departamento",
}


def verificar_unicidad(df, clave, nombre):
    if df[clave].isna().any(axis=1).any():
        raise ValueError(nombre + ": hay nulos en la clave " + str(clave))
    repetidas = df.duplicated(clave, keep=False)
    if repetidas.any():
        raise ValueError(nombre + ": " + str(int(repetidas.sum())) + " filas repiten la clave " + str(clave))


def territorio_de(df):
    # Ciudad en capitales, distrito judicial en provincias.
    valores = []
    for ambito, ciudad, distrito in zip(df["ambito"], df["ciudad"], df["distrito"]):
        if ambito == "capital":
            valores.append(normalizar_geografia(ciudad))
        else:
            valores.append(normalizar_geografia(distrito))
    return pd.Series(valores, index=df.index, dtype="string")


def hashes_originales():
    hashes = {}
    for nombre in FILAS_ORIGINALES:
        for ruta in [PROCESSED / (nombre + ".csv"), PROCESSED / (nombre + ".parquet")]:
            hashes[ruta] = hashlib.sha256(ruta.read_bytes()).hexdigest()
    return hashes

## Leer las doce tablas (y guardar su huella para comprobar al final que no se tocaron)

In [ ]:
hashes_antes = hashes_originales()
originales = {}
for nombre in FILAS_ORIGINALES:
    originales[nombre] = pd.read_parquet(PROCESSED / (nombre + ".parquet"))
observadas = {}
for nombre in originales:
    observadas[nombre] = len(originales[nombre])
if observadas != FILAS_ORIGINALES:
    raise ValueError("Cambió el inventario de tablas originales: " + str(observadas))
print(sum(observadas.values()), "filas en las doce tablas")

## Recursos judiciales por territorio (cuadros 4.1.x)
Capitales (4.1.1): se suman solo las hojas de la jerarquía, por tipo de órgano.
Provincias: se usa la fila TOTALES publicada, después de comprobar que cada una de sus 119 columnas
coincide con la suma de localidades. Juzgados, tribunales, salas y conciliadores quedan separados:
**no existe un "número real de juzgados"**.

In [ ]:
juzgados = originales["juzgados"]
capital = juzgados[(juzgados["cuadro_origen"] == "4.1.1") & (juzgados["es_hoja_jerarquia"] == True) & (juzgados["tipo_columna"] == "otro")].copy()
capital["ambito"] = "capital"
capital["territorio"] = capital["columna_rotulo_canonico"].map(normalizar_geografia)
capital["departamento_derivado"] = capital["territorio"].map(departamento_de_ciudad)
capital["componente"] = capital["tipo_entidad"].map({
    "juzgado": "juzgados_publicados", "tribunal": "tribunales_publicados", "sala": "salas_publicadas",
    "conciliador": "conciliadores_publicados", "otro": "otros_organos_publicados"})
if capital["componente"].isna().any():
    raise ValueError("4.1.1 contiene un tipo de entidad no previsto")
capital_componentes = capital.groupby(["ambito", "territorio", "departamento_derivado", "componente"], sort=False)["valor"].sum(min_count=1)
capital_componentes = capital_componentes.unstack("componente").reset_index()

total_capital = juzgados[(juzgados["cuadro_origen"] == "4.1.1") & (juzgados["fila_id"] == "f037") & (juzgados["tipo_columna"] == "otro")][["columna_rotulo_canonico", "valor"]].copy()
total_capital["territorio"] = total_capital["columna_rotulo_canonico"].map(normalizar_geografia)
total_capital = total_capital.rename(columns={"valor": "total_publicado_fuente"})
capital_componentes = capital_componentes.merge(total_capital[["territorio", "total_publicado_fuente"]], on="territorio", how="left", validate="one_to_one")
presentes = []
for c in COMPONENTES_RECURSOS:
    if c in capital_componentes.columns:
        presentes.append(c)
if not (capital_componentes[presentes].sum(axis=1) == capital_componentes["total_publicado_fuente"]).all():
    raise ValueError("Los componentes hoja de 4.1.1 no cierran con TOTAL GENERAL")
capital_componentes["valor_indeterminado_publicado"] = pd.NA
capital_componentes["celdas_indeterminadas_fuente"] = 0
capital_componentes["clasificacion_recursos_completa"] = True
capital_componentes["cuadros_origen"] = "4.1.1"
capital_componentes["paginas_pdf"] = "109"
capital_componentes["metodo_agregacion"] = "suma_filas_detalle"
capital_componentes["gestion"] = 2023

In [ ]:
provincia = juzgados[juzgados["cuadro_origen"] != "4.1.1"].copy()
comprobaciones = []
for llave, grupo in provincia.groupby(["cuadro_origen", "columna"], sort=False):
    total = grupo[grupo["provincia_o_grupo"] == "TOTALES"]["valor"]
    if len(total) != 1:
        raise ValueError(str(llave) + ": no existe un único TOTALES")
    detalle = grupo[grupo["provincia_o_grupo"] != "TOTALES"]["valor"].sum()
    comprobaciones.append(int(total.iloc[0]) == int(detalle))
if len(comprobaciones) != 119 or not all(comprobaciones):
    raise ValueError("Las filas TOTALES provinciales no reproducen 119/119 columnas")


def componente_provincial(tipo_columna, codigo):
    if tipo_columna == "conciliador":
        return "conciliadores_publicados"
    if tipo_columna == "indeterminado":
        return "valor_indeterminado_publicado"
    if tipo_columna != "organo_judicial":
        return None
    if codigo in CODIGOS_TRIBUNAL:
        return "tribunales_publicados"
    if codigo in CODIGOS_OTRO_ORGANO:
        return "otros_organos_publicados"
    if isinstance(codigo, str) and codigo.startswith("juzgado_"):
        return "juzgados_publicados"
    raise ValueError("Código de órgano provincial no clasificado: " + repr(codigo))


totales = provincia[provincia["provincia_o_grupo"] == "TOTALES"].copy()
totales["ambito"] = "provincia"
totales["territorio"] = totales["departamento"].map(normalizar_geografia)
totales["componente"] = [componente_provincial(t, c) for t, c in zip(totales["tipo_columna"], totales["columna_codigo_canonico"])]
componentes_provincia = totales[totales["componente"].notna()].copy()
componentes_provincia = componentes_provincia.groupby(["ambito", "territorio", "departamento_derivado", "componente"], sort=False)["valor"].sum(min_count=1)
componentes_provincia = componentes_provincia.unstack("componente").reset_index()
totales_publicados = totales[totales["tipo_columna"] == "total"][["territorio", "valor"]].rename(columns={"valor": "total_publicado_fuente"})
componentes_provincia = componentes_provincia.merge(totales_publicados, on="territorio", how="left", validate="one_to_one")
presentes = []
for c in COMPONENTES_RECURSOS + ["valor_indeterminado_publicado"]:
    if c in componentes_provincia.columns:
        presentes.append(c)
if not (componentes_provincia[presentes].sum(axis=1) == componentes_provincia["total_publicado_fuente"]).all():
    raise ValueError("Los componentes provinciales no cierran con el total publicado")

# Tarija 4.1.7/col_09 sigue con encabezado indeterminado: se marca, no se interpreta.
indeterminadas = provincia[(provincia["tipo_columna"] == "indeterminado") & (provincia["provincia_o_grupo"] != "TOTALES")].groupby("departamento_derivado").size()
componentes_provincia["celdas_indeterminadas_fuente"] = componentes_provincia["departamento_derivado"].map(indeterminadas).fillna(0).astype("Int64")
componentes_provincia["clasificacion_recursos_completa"] = componentes_provincia["celdas_indeterminadas_fuente"] == 0
componentes_provincia["cuadros_origen"] = componentes_provincia["departamento_derivado"].map(provincia.groupby("departamento_derivado")["cuadro_origen"].first())
componentes_provincia["paginas_pdf"] = componentes_provincia["departamento_derivado"].map(provincia.groupby("departamento_derivado")["pagina_pdf"].first()).astype("Int64").astype("string")
componentes_provincia["metodo_agregacion"] = "total_distrital_publicado"
componentes_provincia["gestion"] = 2023

In [ ]:
recursos = pd.concat([capital_componentes, componentes_provincia], ignore_index=True, sort=False)
for columna in COMPONENTES_RECURSOS + ["valor_indeterminado_publicado", "celdas_indeterminadas_fuente", "total_publicado_fuente", "gestion"]:
    if columna not in recursos.columns:
        recursos[columna] = pd.NA
    recursos[columna] = pd.array(recursos[columna], dtype="Int64")
recursos["clasificacion_recursos_completa"] = pd.array(recursos["clasificacion_recursos_completa"], dtype="boolean")
orden = ["ambito", "territorio", "departamento_derivado", "gestion", "cuadros_origen", "paginas_pdf", "metodo_agregacion"]
orden = orden + COMPONENTES_RECURSOS + ["valor_indeterminado_publicado", "celdas_indeterminadas_fuente", "total_publicado_fuente", "clasificacion_recursos_completa"]
recursos = recursos[orden].sort_values(["ambito", "territorio"], kind="stable", ignore_index=True)
if len(recursos) != 19:
    raise ValueError("Se esperaban 19 claves de recursos; hay " + str(len(recursos)))
verificar_unicidad(recursos, ["ambito", "territorio"], "recursos_judiciales_geografia")
recursos

In [ ]:
graf = recursos.set_index(recursos["ambito"].str[0].str.upper() + " " + recursos["territorio"])[COMPONENTES_RECURSOS].fillna(0)
graf.plot(kind="barh", stacked=True, figsize=(10, 6))
plt.title("Órganos publicados por territorio (C = capital, P = provincia)")
plt.xlabel("cantidad")
plt.tight_layout()
plt.show()

## Personal por distrito judicial (las nueve filas distritales del 14.1.3)

In [ ]:
personal = originales["personal"]
columnas_personal = ["cuadro_origen", "pagina_pdf", "distrito", "departamento_derivado", "gestion", "items_mujer", "items_varon",
                     "items_acefalias", "items_total", "remun_mujer", "remun_varon", "remun_acefalias", "remun_total"]
personal_geo = personal[(personal["cuadro_origen"] == "14.1.3") & (personal["tipo_fila_derivado"] == "dato") & personal["departamento_derivado"].notna()][columnas_personal].copy()
personal_geo["nivel_geografico"] = "distrito_judicial"
personal_geo["metodo_seleccion"] = "fila_distrital_publicada_14.1.3"
personal_geo = personal_geo.sort_values("departamento_derivado", kind="stable", ignore_index=True)
if len(personal_geo) != 9:
    raise ValueError("Se esperaban 9 filas distritales de personal; hay " + str(len(personal_geo)))
verificar_unicidad(personal_geo, ["departamento_derivado"], "personal_geografia")
personal_geo

## Tabla principal: las 1.997 filas territoriales de detalle
`tipo_elemento_analitico` separa los procesos reales de las filas que son acciones penales o
encabezados padre, usando las auditorías (fragmentos primero, después la clasificación de contexto).

In [ ]:
propuesta = pd.read_csv(AUDITORIA / "propuesta_contexto_tipo_proceso.csv", dtype=str)
propuesta = propuesta[(propuesta["tabla"] == "causas_por_tipo_proceso") & propuesta["tipo_proceso"].notna()][["cuadro_origen", "tipo_proceso", "clasificacion_tipo_proceso"]]
if (propuesta.groupby(["cuadro_origen", "tipo_proceso"])["clasificacion_tipo_proceso"].nunique() > 1).any():
    raise ValueError("La auditoría propone dos clasificaciones para el mismo literal")
propuesta = propuesta.groupby(["cuadro_origen", "tipo_proceso"], as_index=False, sort=False)["clasificacion_tipo_proceso"].first()

fragmentos = pd.read_csv(AUDITORIA / "auditoria_fragmentos_tipo_proceso_completa.csv", dtype=str)
fragmentos = fragmentos[fragmentos["tabla"] == "causas_por_tipo_proceso"].copy()
fragmentos["cuadro_origen"] = fragmentos["cuadro_origen"].str.split(";")
fragmentos = fragmentos.explode("cuadro_origen")[["cuadro_origen", "literal_actual", "tipo_elemento"]]
if (fragmentos.groupby(["cuadro_origen", "literal_actual"])["tipo_elemento"].nunique() > 1).any():
    raise ValueError("La auditoría de fragmentos no define una clasificación única")
fragmentos = fragmentos.groupby(["cuadro_origen", "literal_actual"], as_index=False, sort=False)["tipo_elemento"].first()


def clasificar_tipo_elemento(base):
    resultado = base.reset_index(drop=True).copy()
    resultado["_orden_analitico"] = resultado.index
    resultado = resultado.merge(propuesta, left_on=["cuadro_origen", "tipo_proceso_extraido"], right_on=["cuadro_origen", "tipo_proceso"],
                                how="left", validate="many_to_one", suffixes=("", "_auditoria"))
    resultado = resultado.merge(fragmentos, left_on=["cuadro_origen", "tipo_proceso_extraido"], right_on=["cuadro_origen", "literal_actual"],
                                how="left", validate="many_to_one")
    if resultado["clasificacion_tipo_proceso"].isna().any():
        raise ValueError("Hay filas base sin clasificación en propuesta_contexto_tipo_proceso")
    clases = []
    for i, fila in resultado.iterrows():
        if pd.notna(fila["tipo_elemento"]):
            clases.append(fila["tipo_elemento"])
        elif fila["clasificacion_tipo_proceso"] == "proceso_valido":
            clases.append("proceso")
        elif fila["clasificacion_tipo_proceso"] == "detalle_valido_no_proceso":
            if fila["tipo_proceso"] in ACCIONES_PENALES_AUDITADAS:
                clases.append("accion_penal")
            else:
                clases.append("otro_detalle")
        else:
            raise ValueError("Clasificación no aplicable: " + repr(fila["clasificacion_tipo_proceso"]))
    resultado["tipo_elemento_analitico"] = pd.array(clases, dtype="string")
    resultado = resultado.sort_values("_orden_analitico", kind="stable")
    return resultado["tipo_elemento_analitico"].reset_index(drop=True)


causas = originales["causas_por_tipo_proceso"]
mascara_base = (causas["tipo_fila_derivado"] == "detalle") & ~causas["es_total_nacional"] & causas["tipo_proceso"].notna()
base_fuente = causas.loc[mascara_base].copy()
base = causas.loc[mascara_base].copy().reset_index(drop=True)
if len(base) != 1997:
    raise ValueError("El estrato base dejó de tener 1.997 filas: " + str(len(base)))
base["territorio"] = territorio_de(base)
base["tipo_elemento_analitico"] = clasificar_tipo_elemento(base)
verificar_unicidad(base, CLAVE_PRINCIPAL, "dataset base")
columnas_fuente = list(base.columns)
base["tipo_elemento_analitico"].value_counts()

## Uniones N:1 con recursos y personal (se exige que todas las filas encuentren pareja)

In [ ]:
renombre = {"clasificacion_recursos_completa": "recursos_clasificacion_completa"}
for c in COMPONENTES_RECURSOS:
    renombre[c] = "recurso_" + c
recursos_join = recursos[["ambito", "territorio"] + COMPONENTES_RECURSOS + ["clasificacion_recursos_completa"]].rename(columns=renombre)
base = base.merge(recursos_join, on=["ambito", "territorio"], how="left", validate="many_to_one", sort=False, indicator="_union_recursos")
if not (base["_union_recursos"] == "both").all():
    raise ValueError("Hay territorios base sin recursos")
base = base.drop(columns="_union_recursos")

renombre = {}
for c in personal_geo.columns:
    if c != "departamento_derivado":
        renombre[c] = "personal_" + c
personal_join = personal_geo.rename(columns=renombre)
base = base.merge(personal_join, on="departamento_derivado", how="left", validate="many_to_one", sort=False, indicator="_union_personal")
if not (base["_union_personal"] == "both").all():
    raise ValueError("Hay departamentos base sin el total distrital de personal")
base = base.drop(columns="_union_personal")

if len(base) != 1997:
    raise ValueError("Un enriquecimiento produjo fan-out")
verificar_unicidad(base, CLAVE_PRINCIPAL, "dataset analítico enriquecido")
# Las columnas de origen quedan exactamente iguales.
comparables = []
for c in columnas_fuente:
    if c != "territorio" and c != "tipo_elemento_analitico":
        comparables.append(c)
pd.testing.assert_frame_equal(causas.loc[mascara_base].reset_index(drop=True)[comparables], base[comparables], check_exact=True)
principal = base
print(principal.shape)

## Métricas de las otras cuatro familias, apiladas (no se cruzan con la principal)

In [ ]:
partes = []
for tabla in FAMILIAS_METRICAS:
    parte = originales[tabla].copy()
    parte.insert(0, "familia_metrica", FAMILIAS_METRICAS[tabla])
    parte["territorio"] = territorio_de(parte)
    partes.append(parte)
metricas = pd.concat(partes, ignore_index=True, sort=False)
if len(metricas) != 81907:
    raise ValueError("El hecho longitudinal debería tener 81.907 filas: " + str(len(metricas)))
verificar_unicidad(metricas, CLAVE_LONGITUDINAL, "metricas_tipo_proceso_long")

# Cuánto crecería la principal si se unieran las métricas (relación 1:N): por eso no se unen.
validas = metricas[(metricas["tipo_fila_derivado"] == "detalle") & ~metricas["es_total_nacional"] & metricas["tipo_proceso"].notna()].copy()
claves_base = principal[CLAVE_PRINCIPAL]
vinculadas = validas.merge(claves_base, on=CLAVE_PRINCIPAL, how="left", validate="many_to_one", indicator=True)
claves_metricas = validas.groupby(CLAVE_PRINCIPAL, as_index=False, sort=False, dropna=False).size()[CLAVE_PRINCIPAL]
comunes = claves_metricas.merge(claves_base, on=CLAVE_PRINCIPAL, how="inner", validate="one_to_one")
filas_match = int((vinculadas["_merge"] == "both").sum())
base_sin_match = len(principal) - len(comunes)
relacion = {
    "filas_metricas_validas": len(validas),
    "claves_metricas": len(claves_metricas),
    "claves_comunes": len(comunes),
    "base_sin_match": base_sin_match,
    "claves_metricas_sin_match": len(claves_metricas) - len(comunes),
    "filas_metricas_match": filas_match,
    "filas_metricas_sin_match": int((vinculadas["_merge"] == "left_only").sum()),
    "filas_left_join_potencial": filas_match + base_sin_match,
    "factor_potencial": (filas_match + base_sin_match) / len(principal),
}
relacion

## Movimiento (9.1.x) y gestión 2023 por materia: unión externa 1:1 que conserva las no parejas

In [ ]:
movimiento = originales["causas_movimiento"]
gestion = originales["causas_por_gestion"]
mov = movimiento[(movimiento["eje"] == "materia") & (movimiento["tipo_fila_derivado"] == "dato")].copy()
ges = gestion[(gestion["tipo_fila_derivado"] == "dato") & (gestion["gestion"] == 2023)].copy()
clave = ["ambito", "materia_homologada", "gestion"]
verificar_unicidad(mov, clave, "movimiento por materia 2023")
verificar_unicidad(ges, clave, "gestión por materia 2023")
renombre = {}
for c in mov.columns:
    if c not in clave:
        renombre[c] = c + "_movimiento"
mov = mov.rename(columns=renombre)
renombre = {}
for c in ges.columns:
    if c not in clave:
        renombre[c] = c + "_gestion"
ges = ges.rename(columns=renombre)
mov_ges = mov.merge(ges, on=clave, how="outer", validate="one_to_one", indicator=True, sort=False)
mov_ges["estado_union"] = mov_ges["_merge"].map({"both": "both", "left_only": "solo_movimiento", "right_only": "solo_gestion"}).astype("string")
mov_ges = mov_ges.drop(columns="_merge")
conteos = mov_ges["estado_union"].value_counts().to_dict()
if len(mov_ges) != 56 or conteos != {"both": 32, "solo_movimiento": 12, "solo_gestion": 12}:
    raise ValueError("Movimiento/gestión dejó de reproducir 32/12/12: " + str(conteos))
conteos

## Exportar (y comprobar que CSV y Parquet dicen lo mismo)

In [ ]:
ANALITICO.mkdir(parents=True, exist_ok=True)
salidas = {
    "dataset_analitico_interno": principal,
    "metricas_tipo_proceso_long": metricas,
    "movimiento_gestion_2023": mov_ges,
    "recursos_judiciales_geografia": recursos,
    "personal_geografia": personal_geo,
}
pares_csv_parquet = 0
for nombre in salidas:
    df = salidas[nombre]
    df.to_csv(ANALITICO / (nombre + ".csv"), index=False, encoding="utf-8")
    df.to_parquet(ANALITICO / (nombre + ".parquet"), index=False)
    csv_df = pd.read_csv(ANALITICO / (nombre + ".csv"), low_memory=False)
    parquet_df = pd.read_parquet(ANALITICO / (nombre + ".parquet"))
    if list(csv_df.columns) != list(parquet_df.columns) or len(csv_df) != len(parquet_df):
        raise ValueError("CSV/Parquet incompatibles para " + nombre)
    for columna in csv_df.columns:
        a = csv_df[columna]
        b = parquet_df[columna]
        ambos_nulos = a.isna() & b.isna()
        if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
            iguales = (a == b) | ambos_nulos
        else:
            iguales = (a.astype("string") == b.astype("string")) | ambos_nulos
        if not iguales.all():
            raise ValueError("CSV/Parquet difieren en " + nombre + "." + columna)
    pares_csv_parquet = pares_csv_parquet + 1
    print(nombre, len(df), "filas", len(df.columns), "columnas")

## Diccionario analítico: descripción, rol y riesgo de *leakage* de cada columna
*Leakage*: usar como predictor una variable que es parte del resultado que se quiere explicar.

In [ ]:
GEOGRAFICAS = {"ambito", "territorio", "ciudad", "distrito", "departamento", "departamento_derivado", "localidad_o_subtipo",
               "provincia_o_grupo", "personal_distrito", "personal_nivel_geografico"}
MATERIAS = {"materia_cruda", "materia_norm", "materia_homologada"}
PROCESOS = {"tipo_proceso_extraido", "tipo_proceso", "tipo_accion_penal", "grupo_proceso", "grupo_proceso_norm", "materia_seccion",
            "etapa_proceso_fuente", "contexto_accion_penal", "tipo_elemento_analitico"}
TRAZABILIDAD = {"cuadro_origen", "pagina_pdf", "orden_fila", "firma", "familia", "titulo_pagina", "entidad", "unidad_fila",
                "n_columnas", "revisado_manual", "columna", "orden_columna", "rotulo_columna_pdf", "cuadros_origen", "paginas_pdf",
                "metodo_agregacion", "personal_cuadro_origen", "personal_pagina_pdf", "personal_metodo_seleccion", "metodo_seleccion"}
NO_RESULTADOS = {"pagina_pdf", "orden_fila", "orden_columna", "n_columnas", "gestion", "num_juzgados", "num_juzgados_pagina",
                 "revisado_manual", "errata_corregida", "es_total_nacional", "personal_pagina_pdf", "personal_gestion",
                 "celdas_indeterminadas_fuente", "recursos_clasificacion_completa"}


def sin_sufijo(columna):
    return columna.removesuffix("_movimiento").removesuffix("_gestion")


def empieza_con_alguno(columna, conjunto):
    for c in conjunto:
        if columna.startswith(c + "_"):
            return True
    return False


def rol_columna(tabla, columna):
    if columna in GEOGRAFICAS:
        return "geografia"
    if columna in MATERIAS or empieza_con_alguno(columna, MATERIAS):
        return "materia"
    if columna in PROCESOS or empieza_con_alguno(columna, PROCESOS):
        return "proceso"
    if columna in {"familia_metrica", "estado_union"}:
        return "identificador"
    if sin_sufijo(columna) in {"num_juzgados", "num_juzgados_pagina"}:
        return "no_usar_como_predictor"
    if columna in TRAZABILIDAD or empieza_con_alguno(columna, TRAZABILIDAD):
        return "trazabilidad"
    if columna.startswith(("recurso_", "recursos_", "personal_")) or tabla in {"recursos_judiciales_geografia", "personal_geografia"}:
        return "recurso"
    return "trazabilidad"


def es_resultado(tabla, columna, serie):
    if not pd.api.types.is_numeric_dtype(serie):
        return False
    if columna in NO_RESULTADOS or sin_sufijo(columna) in NO_RESULTADOS:
        return False
    if tabla in {"recursos_judiciales_geografia", "personal_geografia"}:
        return False
    if columna.startswith(("recurso_", "recursos_", "personal_")):
        return False
    return True


fuente = pd.read_csv(PROCESSED / "diccionario_de_datos.csv", dtype=str)
descripciones = {}
for i, r in fuente.iterrows():
    descripciones[(r["tabla"], r["columna"])] = r["descripcion"]
PERSONALIZADOS = {
    "territorio": "Clave geográfica comparable: ciudad para capital y El Alto; distrito judicial para provincia.",
    "tipo_elemento_analitico": "Clasificación analítica auditada: proceso, acción penal u otro detalle; no modifica el literal fuente.",
    "familia_metrica": "Familia de la tabla longitudinal de procedencia.",
    "estado_union": "Resultado trazable del outer join movimiento/gestión.",
    "metodo_agregacion": "Regla auditada usada para construir el recurso.",
    "metodo_seleccion": "Regla auditada de selección de la fila distrital.",
    "nivel_geografico": "Nivel territorial al que corresponde el recurso.",
    "cuadros_origen": "Cuadro o cuadros fuente del recurso agregado.",
    "paginas_pdf": "Página o páginas fuente del recurso agregado.",
    "clasificacion_recursos_completa": "False cuando existe una columna de recurso cuyo encabezado sigue indeterminado.",
    "celdas_indeterminadas_fuente": "Número de celdas territoriales con encabezado indeterminado.",
    "valor_indeterminado_publicado": "Total publicado de una columna no interpretada; no se incorpora como componente al dataset principal.",
    "total_publicado_fuente": "Total impreso usado solo para validar componentes; no equivale a numero_juzgados_real.",
}
ORIGEN_TABLA = {"dataset_analitico_interno": "causas_por_tipo_proceso", "personal_geografia": "personal"}


def describir_columna(tabla, columna):
    d = PERSONALIZADOS.get(columna)
    if d is None:
        d = descripciones.get((ORIGEN_TABLA.get(tabla), columna))
    if d is None and tabla == "metricas_tipo_proceso_long":
        for t in FAMILIAS_METRICAS:
            if d is None and descripciones.get((t, columna)):
                d = descripciones.get((t, columna))
    if d is None and tabla == "movimiento_gestion_2023":
        if columna.endswith("_movimiento"):
            d = descripciones.get(("causas_movimiento", columna.removesuffix("_movimiento")))
        elif columna.endswith("_gestion"):
            d = descripciones.get(("causas_por_gestion", columna.removesuffix("_gestion")))
        else:
            d = descripciones.get(("causas_por_gestion", columna))
    if d is None and tabla == "recursos_judiciales_geografia" and columna.endswith("_publicados"):
        d = "Suma o total publicado del componente " + columna.removesuffix("_publicados").replace("_", " ") + "; no es un total general de juzgados."
    if d is None and tabla == "dataset_analitico_interno":
        if columna.startswith("recurso_"):
            d = "Componente geográfico derivado de los cuadros 4.1.x; no se asigna por materia o proceso."
        elif columna.startswith("personal_"):
            d = descripciones.get(("personal", columna.removeprefix("personal_")))
    if d is None and tabla == "personal_geografia":
        d = descripciones.get(("personal", columna))
    if d is None:
        d = "Campo conservado o derivado por la integración interna; ver documentación de la tabla."
    return d


filas = []
for tabla in salidas:
    df = salidas[tabla]
    for columna in df.columns:
        rol = rol_columna(tabla, columna)
        leakage = es_resultado(tabla, columna, df[columna])
        if leakage:
            rol = "resultado"
        origen = "derivada en integración"
        if tabla == "dataset_analitico_interno" and columna in causas.columns:
            origen = "causas_por_tipo_proceso"
        elif tabla == "metricas_tipo_proceso_long" and columna not in {"familia_metrica", "territorio"}:
            origen = "tabla de familia_metrica"
        elif tabla == "movimiento_gestion_2023":
            origen = "causas_movimiento / causas_por_gestion"
        elif tabla == "recursos_judiciales_geografia":
            origen = "juzgados"
        elif tabla == "personal_geografia":
            origen = "personal / cuadro 14.1.3"
        observacion = ""
        if leakage:
            observacion = "Resultado o componente del resultado: no usar como predictor del mismo outcome."
        if "remun" in columna:
            observacion = "Recurso estructural; no usar como predictor si el outcome se deriva de remuneraciones o costo."
        if columna in {"num_juzgados", "num_juzgados_pagina"}:
            observacion = "Conteo nominal/de página; no sustituye un número físico auditado de juzgados."
        filas.append({"tabla": tabla, "columna": columna, "descripcion": " ".join(describir_columna(tabla, columna).split()),
                      "rol_analitico": rol, "origen": origen, "grano": GRANOS_ANALITICOS[tabla], "riesgo_leakage": leakage,
                      "observaciones": observacion})
diccionario_analitico = pd.DataFrame(filas)
diccionario_analitico.to_csv(ANALITICO / "diccionario_analitico.csv", index=False, encoding="utf-8")
diccionario_analitico.groupby(["tabla", "rol_analitico"]).size().unstack(fill_value=0)

## Controles finales de integración

In [ ]:
hashes_despues = hashes_originales()
intactos = 0
for ruta in hashes_antes:
    if hashes_antes[ruta] == hashes_despues[ruta]:
        intactos = intactos + 1

controles = []


def control_integracion(nombre, esperado, observado):
    if esperado == observado:
        estado = "OK"
    else:
        estado = "ERROR"
    controles.append({"control": nombre, "esperado": esperado, "observado": observado, "estado": estado})


control_integracion("tablas originales", 12, len(originales))
control_integracion("filas tablas originales", 85953, sum(observadas.values()))
control_integracion("archivos originales intactos", 24, intactos)
control_integracion("filas estrato base", 1997, len(base_fuente))
control_integracion("filas dataset principal", 1997, len(principal))
control_integracion("claves únicas dataset principal", 1997, principal.groupby(CLAVE_PRINCIPAL, dropna=False).ngroups)
control_integracion("duplicados clave principal", 0, int(principal.duplicated(CLAVE_PRINCIPAL).sum()))
control_integracion("fan-out dataset principal", 1.0, len(principal) / len(base_fuente))
control_integracion("joins N:1 válidos", 2, 2)
control_integracion("joins N:M persistidos", 0, 0)
control_integracion("pares CSV/Parquet equivalentes", 5, pares_csv_parquet)
control_integracion("claves únicas recursos", 19, recursos.groupby(["ambito", "territorio"], dropna=False).ngroups)
control_integracion("claves únicas personal", 9, personal_geo["departamento_derivado"].nunique())
control_integracion("filas métricas longitudinales", 81907, len(metricas))
control_integracion("claves técnicas métricas", 81907, metricas.groupby(CLAVE_LONGITUDINAL, dropna=False).ngroups)
control_integracion("claves base con métricas", 1712, relacion["claves_comunes"])
control_integracion("filas movimiento gestión 2023", 56, len(mov_ges))
for estado, esperado in [("both", 32), ("solo_movimiento", 12), ("solo_gestion", 12)]:
    control_integracion("movimiento estado " + estado, esperado, int((mov_ges["estado_union"] == estado).sum()))
for metrica in SUMAS_BASE_ESPERADAS:
    control_integracion("suma base " + metrica, SUMAS_BASE_ESPERADAS[metrica], int(principal[metrica].sum()))
validacion = pd.DataFrame(controles)
if not (validacion["estado"] == "OK").all():
    raise ValueError("Fallaron controles de integración:\n" + validacion[validacion["estado"] != "OK"].to_string(index=False))
validacion.to_csv(AUDITORIA / "validacion_integracion_interna.csv", index=False, encoding="utf-8")

cobertura = pd.DataFrame([
    {"fuente": "recursos_judiciales_geografia", "filas_base": len(principal), "matches": len(principal), "sin_match": 0,
     "porcentaje_match": 100.0, "cardinalidad": "N:1", "factor_expansion": 1.0, "decision": "incorporado"},
    {"fuente": "personal_geografia", "filas_base": len(principal), "matches": len(principal), "sin_match": 0,
     "porcentaje_match": 100.0, "cardinalidad": "N:1", "factor_expansion": 1.0, "decision": "incorporado"},
    {"fuente": "metricas_tipo_proceso_long", "filas_base": len(principal), "matches": relacion["claves_comunes"],
     "sin_match": relacion["base_sin_match"], "porcentaje_match": round(100 * relacion["claves_comunes"] / len(principal), 6),
     "cardinalidad": "1:N", "factor_expansion": round(relacion["factor_potencial"], 6), "decision": "hecho_auxiliar_no_incorporado"},
])
cobertura.to_csv(AUDITORIA / "cobertura_integracion_interna.csv", index=False, encoding="utf-8")
validacion